# Experiment 2: Metadata 
This second experiment consists on seeing which is the best technique concerning metadata: 
- No using metadata
- Using metadata and concatenate it to the text embeddings 
- Using metadata with it's separate embeddings and have a score on both embeddings

In [1]:
import os, json
import google.generativeai as genai
import uuid
import fitz  #pip install pymupdf
import json
import tempfile
import requests
import numpy as np
from pathlib import Path
import sys
import requests

from embedder import Embedder

C:\Users\lucia\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\lucia\AppData\Local\Temp\ipykernel_46688\4249173254.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:


# we are in: backend/exp2/experiment_2_metadata.ipynb
BASE_DIR = Path.cwd().parents[0]   # backend/
DATA_DIR = BASE_DIR / "data"
PDF_DIR = DATA_DIR / "pdfs"
DB_PATH = DATA_DIR / "database.json"

sys.path.append(str(BASE_DIR))

print("PDF_DIR:", PDF_DIR)
print("PDFs:", len(list(PDF_DIR.glob("*.pdf"))))
print("DB exists:", DB_PATH.exists())

PDF_DIR: c:\Lucía\Lucia\uni\z otras cosas\ERASMUS\VIENA\asignaturas\GenAI\GenAI-PR-2025w\backend\data\pdfs
PDFs: 15
DB exists: True


In [3]:
with open(DB_PATH, "r", encoding="utf-8") as f:
    db = json.load(f)

pdf_names = [e["pdf_name"] for e in db if "pdf_name" in e]
pdf_paths = [PDF_DIR / name for name in pdf_names if (PDF_DIR / name).exists()]

print("PDFs in DB:", len(pdf_names))
print("PDFs found on disk:", len(pdf_paths))
pdf_paths[:3]

PDFs in DB: 15
PDFs found on disk: 15


[WindowsPath('c:/Lucía/Lucia/uni/z otras cosas/ERASMUS/VIENA/asignaturas/GenAI/GenAI-PR-2025w/backend/data/pdfs/1907.02052v1.pdf'),
 WindowsPath('c:/Lucía/Lucia/uni/z otras cosas/ERASMUS/VIENA/asignaturas/GenAI/GenAI-PR-2025w/backend/data/pdfs/1911.00536v3.pdf'),
 WindowsPath('c:/Lucía/Lucia/uni/z otras cosas/ERASMUS/VIENA/asignaturas/GenAI/GenAI-PR-2025w/backend/data/pdfs/1911.02365v1.pdf')]

In [4]:
#gemini metadata 

#EVERYONE NEEDS THEIR API KEY
#environment variable
#Windows powershell => setx GEMINI_API_KEY "YOUR_API_KEY"
#Linux/MacOs => export GEMINI_API_KEY="YOUR_API_KEY"
os.environ["GEMINI_API_KEY"] = "ATATA"
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("Missing GEMINI_API_KEY env var")


genai.configure(api_key=api_key)

#GEMINI_MODEL = "gemini-2.0-flash" 
GEMINI_MODEL = "gemini-2.0-flash-lite"

META_CACHE_PATH = DATA_DIR / "llm_metadata_cache.json"

if META_CACHE_PATH.exists():
    metadata_cache = json.loads(META_CACHE_PATH.read_text(encoding="utf-8"))
else:
    metadata_cache = {}


def extract_llm_metadata(doc_text):
    """
    Returns metadata in JSON from the text of the document
    """

    #client = genai.Client(api_key=api_key)

    # recorta para no pasarle el documento entero (suficiente con inicio/abstract)
    snippet = doc_text[:12000]

    prompt = f"""
        You are extracting bibliographic and topical metadata from a PDF text dump.
        Return ONLY valid JSON (no markdown).

        Schema:
        {{
        "title": string|null,
        "authors": [string],
        "year": int|null,
        "keywords": [string],   // 8-15 items
        "topics": [string],     // 2-5 short tags
        "one_sentence_summary": string|null
        }}

        Rules:
        - If you are unsure, use null or empty lists.
        - Keep keywords/topics concise (1-4 words).
        - Do not hallucinate specific author names if not present.
        - Base everything only on the provided text.

        TEXT:
        {snippet}
        """.strip()

    url = (
        "https://generativelanguage.googleapis.com/v1beta/"
        "models/gemini-2.5-flash:generateContent"
    )

    headers = {
        "Content-Type": "application/json",
        "x-goog-api-key": api_key
    }

    payload = {
        "contents": [
            {"parts": [{"text": prompt}]}
        ]
    }

    response = requests.post(url, json=payload, headers=headers)
    response.raise_for_status()
    data = response.json()

    raw = data["candidates"][0]["content"]["parts"][0]["text"]

    # quitar ```json y ```
    raw = raw.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print("\n=== JSON PARSE FAILED ===")
        print("Error:", e)
        print("RAW OUTPUT:\n", raw)
        print("========================\n")
        raise
    #text = data["candidates"][0]["content"]["parts"][0]["text"]
    
    return json.loads(raw)


import time
def get_llm_metadata(pdf_name, doc_text):
    if pdf_name not in metadata_cache:
        time.sleep(4)
        metadata_cache[pdf_name] = extract_llm_metadata(doc_text)
        META_CACHE_PATH.write_text(json.dumps(metadata_cache, indent=2), encoding="utf-8")
    return metadata_cache[pdf_name]

def build_metadata_string(llm_meta):

    if not llm_meta:
        return ""
    parts = []
    if llm_meta.get("title"):
        parts.append(f"Title: {llm_meta['title']}")
    if llm_meta.get("topics"):
        parts.append("Topics: " + ", ".join(llm_meta["topics"]))
    if llm_meta.get("keywords"):
        parts.append("Keywords: " + ", ".join(llm_meta["keywords"]))
    if llm_meta.get("one_sentence_summary"):
        parts.append("Summary: " + llm_meta["one_sentence_summary"])
    return "[METADATA]\n" + "\n".join(parts)

In [41]:
# data_manager.py

class DataManager:
    """
    Handles everything related to:
    - downloading or receiving PDFs
    - saving them to the database
    - extracting text
    - chunking, embedding & storing database
    """

    def __init__(self, database_file=DB_PATH, pdf_folder=PDF_DIR, metadata_mode = "concat"):
        self.embedder = Embedder()
        self.database_file = database_file
        self.pdf_folder = pdf_folder
        #self.database = self.load_database()
        self.metadata_mode = metadata_mode #"concat" or "dual"

    # ------------------------------
    # DATABASE I/O
    # ------------------------------
    '''    def load_database(self):
            if os.path.exists(self.database_file):
                with open(self.database_file, "r", encoding="utf-8") as f:
                    return json.load(f)
            return []

        def save_database(self):
            with open(self.database_file, "w", encoding="utf-8") as f:
                json.dump(self.database, f, indent=4, ensure_ascii=False)
    '''
    # ------------------------------
    # INTERNAL UTILITIES
    # ------------------------------
    '''def _save_pdf_to_db(self, file_path, arxiv_id=None):
        """Assigns a DB filename and copies the PDF inside /data/pdfs."""
        pdf_name = f"{arxiv_id}.pdf" if arxiv_id else f"{uuid.uuid4()}.pdf"
        out_path = os.path.join(self.pdf_folder, pdf_name)
        with open(file_path, "rb") as src, open(out_path, "wb") as dst:
            dst.write(src.read())
        return pdf_name

    def _create_database_entry(self, title, pdf_name, researcher):
        entry = {
            "id": str(uuid.uuid4()),
            "title": title,
            "pdf_name": pdf_name,
            "researcher": researcher,
            "chunks": []  # filled after processing
        }
        self.database.append(entry)
        self.save_database()
        return entry
    

    # ------------------------------
    # MAIN UPLOAD METHODS
    # ------------------------------

    def upload_pdf(self, file_path, title, researcher, arxiv_id):
        """UPLOAD from a local file path and index it immediately."""
        self.database = self.load_database()
        pdf_name = self._save_pdf_to_db(file_path, arxiv_id)
        entry = self._create_database_entry(title, pdf_name, researcher)
        self.process_pdf(entry)
        print(f"Uploaded & indexed: {title}")
        return entry'''
    
    # ------------------------------
    # METADATA
    # ------------------------------
    '''def build_metadata_string(self, entry): #entry is dict
        m = entry.get("llm_metadata") or {}
        parts = []
        if m.get("title"):
            parts.append(f"Title: {m['title']}")
        if m.get("authors"):
            parts.append("Authors: " + ", ".join(m["authors"][:10]))
        if m.get("year"):
            parts.append(f"Year: {m['year']}")
        if m.get("topics"):
            parts.append("Topics: " + ", ".join(m["topics"]))
        if m.get("keywords"):
            parts.append("Keywords: " + ", ".join(m["keywords"]))
        if m.get("one_sentence_summary"):
            parts.append("Summary: " + m["one_sentence_summary"])

        if not parts:
            return ""
        
        META_CACHE = DATA_DIR / "llm_metadata_cache.json"

        if META_CACHE.exists():
            with open(META_CACHE, "r", encoding="utf-8") as f:
                metadata_cache = json.load(f)
        else:
            metadata_cache = {}

        return "[METADATA]\n" + "\n".join(parts)


    def ensure_llm_metadata(self, entry, doc_text):
        """
        Adds entry['llm_metadata'] if missing. Calls Gemini once per document.
        """
        if entry.get("llm_metadata") is not None:
            return  # already present (even if empty dict)

        try:
            entry["llm_metadata"] = extract_llm_metadata(doc_text)
            self.save_database()  # persist early to avoid repeated calls
        except Exception as e:
            print("Gemini metadata extraction failed:", e)
            entry["llm_metadata"] = {}
            self.save_database()  '''    

    # ------------------------------
    # PROCESSING (CHUNK + EMBEDDING)
    # ------------------------------
    def extract_text(self, pdf_path):
        doc = fitz.open(pdf_path)
        text = "".join([page.get_text() for page in doc])
        doc.close()
        return text

    def chunk_text(self, text, max_chars=1000):
        return [text[i:i + max_chars] for i in range(0, len(text), max_chars)]


    def process_pdf(self, pdf_path, mode = "baseline"):
        '''pdf_path = os.path.join(self.pdf_folder, entry["pdf_name"])
        if not os.path.exists(pdf_path):
            print("PDF missing:", pdf_path)
            return'''

        text = self.extract_text(pdf_path)
        chunks = self.chunk_text(text)

        llm_meta = {}
        meta_str = ""
        if mode in ("concat", "dual"):
            llm_meta = get_llm_metadata(pdf_path.name, text) or {}
            meta_str = build_metadata_string(llm_meta)
    
        #metadata doc-level
        '''self.ensure_llm_metadata(entry, text)
        metadata_str = self.build_metadata_string(entry)
        '''
        if mode == "dual": 
            emb_meta = self.embedder.encode(meta_str)

        chunk_records = []
        for chunk in chunks: 
            if mode == "baseline":
                emb = self.embedder.encode(chunk)
                chunk_records.append({"text": chunk, "embedding": emb})
            elif mode == "concat":
                emb = self.embedder.encode(chunk + "\n\n" + meta_str)
                chunk_records.append({"text": chunk, "embedding": emb})
            elif mode == "dual":
                emb_text = self.embedder.encode(chunk)
                chunk_records.append({"text": chunk, "embedding_text": emb_text, "embedding_meta": emb_meta})
        
        return {"pdf_name": pdf_path.name, "llm_metadata": llm_meta, "chunks": chunk_records}
        '''for chunk in chunks:
            if mode == "concat":
                text_to_embed = chunk + "\n\n" + metadata_str if metadata_str else chunk
                embedding = self.embedder.encode(text_to_embed)

            entry["chunks"].append({
                "id": str(uuid.uuid4()),
                "text": chunk,
                "embedding": embedding
            })

        self.save_database()
        print(f"Indexed {len(chunks)} chunks for: {entry['title']}")
        return entry'''

    def build_index(self, pdf_paths, mode = "baseline"):
        self.database = []
        for p in pdf_paths:
            entry = self.process_pdf(p, mode=mode)
            print("PDF processed")
            self.database.append(entry)
        return self.database

In [57]:
#baseline retriever

class BaselineRetriever:
    def __init__(self, database = None, database_file=DB_PATH, embedder = None, mode = "not_dual"):
        """
        database: list (in-memory database). If provided, we won't load from file.
        database_file: path to JSON, used if database is None.
        embedder: optional shared Embedder instance.
        """
        self.embedder = embedder or Embedder()
        self.database_file = database_file
        self.database = database #can be None or list
        self.mode = mode #not_dual or dual

    def load_database(self):
        if self.database is not None:
            return self.database
        if os.path.exists(self.database_file):
            with open(self.database_file, "r", encoding="utf-8") as f:
                return json.load(f)
        return []

    def cosine_similarity(self, v1, v2):
        v1 = np.array(v1)
        v2 = np.array(v2)
        return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

    def search(self, query, threshold=0.70, alpha = 0.7):
        """Returns papers ranked by similarity. threshold ~ 0.65-0.80 recommended"""    
        self.database = self.load_database()

        if len(self.database) == 0:
            print("Database is empty: no entries to search.\n")

        query_emb = self.embedder.encode(query)
        results = []

        for entry in self.database:
            if "chunks" not in entry:
                continue  # not processed yet
            best_score = 0
            best_chunk = None

            for chunk in entry["chunks"]:
                if self.mode == "dual":
                    if chunk.get("embedding_text") is None or chunk.get("embedding_meta") is None:
                        continue
                    s_text = self.cosine_similarity(query_emb, chunk["embedding_text"])
                    s_meta = self.cosine_similarity(query_emb, chunk["embedding_meta"])

                    score = alpha * s_text + (1 - alpha) * s_meta
                else:
                    score = self.cosine_similarity(query_emb, chunk["embedding"])
                
                if score > best_score:
                    best_score = score
                    best_chunk = chunk
            
            if best_score >= threshold:
                results.append({
                "paper_id": entry.get("id", entry.get("pdf_name", "")),
                "title": (entry.get("llm_metadata") or {}).get("title", entry.get("pdf_name", "")), #"title": entry.get("llm_metadata", {}).get("title", entry.get("pdf_name")),
                "pdf_name": entry.get("pdf_name", ""),
                "score": round(best_score, 3),
                "sample_text": best_chunk["text"][:300] if best_chunk else ""
})
        # Sort best match → worst
        results.sort(key=lambda x: x["score"], reverse=True)
        return results

In [31]:
embedder = Embedder()

#dm = DataManager(embedder = embedder, max_chars = 1000)
dm = DataManager()

pdf_paths = list(PDF_DIR.glob("*.pdf"))

import random
random.seed(0)
subset_paths = random.sample(pdf_paths, k=15)

print("PDFs to index:", len(subset_paths))

PDFs to index: 15


In [32]:
baseline_db = dm.build_index(subset_paths, mode="baseline")

PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed


In [9]:
baseline_db[0]["llm_metadata"]

{}

In [44]:
concat_db = dm.build_index(subset_paths, mode="concat")

PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed


In [45]:
concat_db[0]["llm_metadata"]

{'title': 'GPT Understands, Too',
 'authors': ['Xiao Liu',
  'Yanan Zheng',
  'Zhengxiao Du',
  'Ming Ding',
  'Yujie Qian',
  'Zhilin Yang',
  'Jie Tang'],
 'year': 2023,
 'keywords': ['P-Tuning',
  'prompting',
  'natural language understanding',
  'NLU',
  'continuous prompts',
  'discrete prompts',
  'pretrained language models',
  'PLMs',
  'language model adaptation',
  'LAMA',
  'SuperGLUE',
  'prompt embeddings',
  'training stability',
  'few-shot learning',
  'knowledge probing'],
 'topics': ['Natural Language Processing',
  'Language Models',
  'Prompt Engineering',
  'Machine Learning'],
 'one_sentence_summary': 'P-Tuning is a novel method that employs trainable continuous prompt embeddings to stabilize training and improve performance on various natural language understanding tasks, addressing the instability of manual discrete prompts.'}

In [12]:
concat_db[1]["llm_metadata"]

{'title': 'DARE: Data Augmented Relation Extraction with GPT-2',
 'authors': ['Yannis Papanikolaou', 'Andrea Pierleoni'],
 'year': 2020,
 'keywords': ['Relation Extraction',
  'Data Augmentation',
  'GPT-2',
  'BERT models',
  'Training data generation',
  'Class imbalance',
  'Biomedical datasets',
  'Transformer architectures',
  'Language models',
  'Semantic relations',
  'Natural Language Understanding',
  'Text classification',
  'Oversampling techniques',
  'Supervised learning',
  'Knowledge graphs'],
 'topics': ['Natural Language Processing',
  'Relation Extraction',
  'Data Augmentation',
  'Generative Models'],
 'one_sentence_summary': 'This work introduces DARE, a method using fine-tuned GPT-2 to generate augmented training data for Relation Extraction, effectively addressing limited data and class imbalance for BERT-based classifiers and achieving state-of-the-art results on biomedical datasets.'}

In [46]:
dual_db = dm.build_index(subset_paths, mode="dual")

PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed
PDF processed


In [12]:
baseline_retriever = BaselineRetriever(database = baseline_db, embedder = embedder)
#concat_retriever   = BaselineRetriever(concat_db, embedder)
#dual_retriever     = BaselineRetriever(dual_db, embedder) 

In [19]:
baseline_retriever.search("GPT-2 fine-tuning", threshold=0.6)

[{'paper_id': '2004.13845v1.pdf',
  'title': '2004.13845v1.pdf',
  'researcher': '',
  'pdf_name': '2004.13845v1.pdf',
  'score': 0.667,
  'sample_text': 'effective leading to\nworse results than just using gold data, primarily\nbecause frequent classes ”inﬂuenced” more GPT-\n2 and the model was generating many incorrectly\nlabeled samples.\n4\nExperimental Evaluation\nIn this section we present the empirical evalua-\ntion of our method.\nWe ﬁrst describe the '},
 {'paper_id': '2003.02498v1.pdf',
  'title': '2003.02498v1.pdf',
  'researcher': '',
  'pdf_name': '2003.02498v1.pdf',
  'score': 0.627,
  'sample_text': 'D\nPPL\n(a) Performances of fine-tuned GPT-2 (124M) on validation set\nTop-k sampling with k = 1\n0.79\n7.6\n9.81\n0.62\n0.39\n0.51\nk = 3\n0.76\n7.9\n8.29\n0.70\n0.37\n0.52\nk = 5\n0.7\n8.0\n7.81\n0.75\n0.36\n0.53\nk = 10\n0.74\n8.3\n7.42\n0.83\n0.35\n0.53\nk = 30\n0.71\n8.7\n7.15\n0.94\n0.34\n0.54\n(b) Performances of different Recipe'},
 {'paper_id': '1911.00536v3.pdf',
 

In [ ]:
baseline_retriever.search("cows and milk", threshold=0.6)

[]

In [47]:
concat_retriever   = BaselineRetriever(database = concat_db, embedder = embedder)

In [48]:
concat_retriever.search("GPT-2 fine-tuning", threshold=0.6)

[{'paper_id': '2004.13845v1.pdf',
  'title': 'DARE: Data Augmented Relation Extraction with GPT-2',
  'pdf_name': '2004.13845v1.pdf',
  'score': 0.666,
  'sample_text': 'effective leading to\nworse results than just using gold data, primarily\nbecause frequent classes ”inﬂuenced” more GPT-\n2 and the model was generating many incorrectly\nlabeled samples.\n4\nExperimental Evaluation\nIn this section we present the empirical evalua-\ntion of our method.\nWe ﬁrst describe the '},
 {'paper_id': '2003.02498v1.pdf',
  'title': 'RecipeGPT: Generative Pre-training Based Cooking Recipe Generation and Evaluation System',
  'pdf_name': '2003.02498v1.pdf',
  'score': 0.627,
  'sample_text': 'D\nPPL\n(a) Performances of fine-tuned GPT-2 (124M) on validation set\nTop-k sampling with k = 1\n0.79\n7.6\n9.81\n0.62\n0.39\n0.51\nk = 3\n0.76\n7.9\n8.29\n0.70\n0.37\n0.52\nk = 5\n0.7\n8.0\n7.81\n0.75\n0.36\n0.53\nk = 10\n0.74\n8.3\n7.42\n0.83\n0.35\n0.53\nk = 30\n0.71\n8.7\n7.15\n0.94\n0.34\n0.54\n(b) Perf

In [49]:
concat_retriever.search("cows and milk", threshold=0.6)

[]

In [26]:
print(baseline_retriever.search("prompt tuning method for NLU", threshold=0.6))
print(concat_retriever.search("prompt tuning method for NLU", threshold=0.6))

[{'paper_id': '2103.10385v2.pdf', 'title': '2103.10385v2.pdf', 'pdf_name': '2103.10385v2.pdf', 'score': 0.658, 'sample_text': 'mance of automatically\nsearched prompts and P-Tuning. We evaluated LM-\nBFF (Auto) using the reported top-3 searched patterns\nunder our evaluation procedure. P-Tuning also uses\nthe same discrete prompts, in concatenation with con-\ntinuous prompts. Results show that P-Tuning can be\neffectively combine'}]
[{'paper_id': '2103.10385v2.pdf', 'title': 'GPT Understands, Too', 'pdf_name': '2103.10385v2.pdf', 'score': 0.609, 'sample_text': 'mance of automatically\nsearched prompts and P-Tuning. We evaluated LM-\nBFF (Auto) using the reported top-3 searched patterns\nunder our evaluation procedure. P-Tuning also uses\nthe same discrete prompts, in concatenation with con-\ntinuous prompts. Results show that P-Tuning can be\neffectively combine'}]


In [56]:
dual_retriever = BaselineRetriever(database = dual_db, embedder = embedder,  mode = "dual")
dual_retriever.search("GPT-2 fine-tuning", threshold=0.6)

[]

In [59]:
def debug_dual_scores(retriever, query, alpha=0.7, top_k=5):
    db = retriever.load_database()
    q = retriever.embedder.encode(query)
    scored = []
    for entry in db:
        # best text chunk
        best_text = -1
        for ch in entry["chunks"]:
            s_text = retriever.cosine_similarity(q, ch["embedding_text"])
            if s_text > best_text:
                best_text = s_text

        # meta (constante por doc, la leemos del primer chunk)
        s_meta = retriever.cosine_similarity(q, entry["chunks"][0]["embedding_meta"])

        score = alpha * best_text + (1 - alpha) * s_meta
        scored.append((score, best_text, s_meta, entry["pdf_name"]))

    scored.sort(reverse=True, key=lambda x: x[0])
    return scored[:top_k]


In [60]:
debug_dual_scores(dual_retriever, "GPT-2 fine-tuning", alpha=0.7, top_k=10)


[(0.5684819487445962,
  0.6670132861938133,
  0.3385754946964233,
  '2004.13845v1.pdf'),
 (0.5248408285741114,
  0.5898312243995742,
  0.37319657164803155,
  '2005.09123v2.pdf'),
 (0.5189633259685766,
  0.6272259762863689,
  0.2663504752270614,
  '2003.02498v1.pdf'),
 (0.5041935887426557,
  0.598273842374838,
  0.28467299693423054,
  '2007.00659v2.pdf'),
 (0.4933552060550358,
  0.5145566940308716,
  0.4438850674447523,
  '2103.10385v2.pdf'),
 (0.4854584464148878,
  0.5456083699620495,
  0.34510862480484367,
  '2104.04466v3.pdf'),
 (0.48070904620422816,
  0.54679958853038,
  0.3264977807765404,
  '2006.15437v1.pdf'),
 (0.480493724303505,
  0.5657040727420138,
  0.28166957794698433,
  '2102.08036v1.pdf'),
 (0.44685139269080904,
  0.6168674118031852,
  0.050147348095264783,
  '1911.00536v3.pdf'),
 (0.38813976965510844,
  0.5329936646093272,
  0.050147348095264783,
  '1907.02052v1.pdf')]